In [1]:
#importing the Libraies
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Reading the Dataset
dataset = pd.read_csv('Social_Network_Ads.csv')

In [3]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [4]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [5]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [6]:
indep=dataset[['Age', 'EstimatedSalary','Gender_Male']]
dep=dataset['Purchased']

In [7]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)


In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [9]:
from sklearn.linear_model import LogisticRegression

In [10]:
#https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {'solver':['newton-cg', 'lbfgs', 'liblinear', 'saga'],
             'penalty':['l2']} 



grid = GridSearchCV(LogisticRegression(), param_grid, refit = True, verbose = 3,n_jobs=1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train, y_train) 
 



Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV 1/5] END ......penalty=l2, solver=newton-cg;, score=0.800 total time=   0.0s
[CV 2/5] END ......penalty=l2, solver=newton-cg;, score=0.811 total time=   0.0s
[CV 3/5] END ......penalty=l2, solver=newton-cg;, score=0.682 total time=   0.0s
[CV 4/5] END ......penalty=l2, solver=newton-cg;, score=0.924 total time=   0.0s
[CV 5/5] END ......penalty=l2, solver=newton-cg;, score=0.922 total time=   0.0s
[CV 1/5] END ..........penalty=l2, solver=lbfgs;, score=0.800 total time=   0.0s
[CV 2/5] END ..........penalty=l2, solver=lbfgs;, score=0.811 total time=   0.0s
[CV 3/5] END ..........penalty=l2, solver=lbfgs;, score=0.682 total time=   0.0s
[CV 4/5] END ..........penalty=l2, solver=lbfgs;, score=0.924 total time=   0.0s
[CV 5/5] END ..........penalty=l2, solver=lbfgs;, score=0.922 total time=   0.0s
[CV 1/5] END ......penalty=l2, solver=liblinear;, score=0.805 total time=   0.0s
[CV 2/5] END ......penalty=l2, solver=liblinear;,

GridSearchCV(estimator=LogisticRegression(), n_jobs=1,
             param_grid={'penalty': ['l2'],
                         'solver': ['newton-cg', 'lbfgs', 'liblinear', 'saga']},
             scoring='f1_weighted', verbose=3)

In [13]:
# print best parameter after tuning 
#print(grid.best_params_) 
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(X_test) 
   

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)



# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)




In [14]:

from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'penalty': 'l2', 'solver': 'liblinear'}: 0.8794289739855382


In [15]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[79  6]
 [10 39]]


In [16]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.89      0.93      0.91        85
           1       0.87      0.80      0.83        49

    accuracy                           0.88       134
   macro avg       0.88      0.86      0.87       134
weighted avg       0.88      0.88      0.88       134



In [17]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])


0.9507803121248499

In [18]:
table=pd.DataFrame.from_dict(re)

In [19]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.009903,0.008583,0.002081,0.001554,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.799620,0.811321,0.68205,0.92351,0.922185,0.827737,0.089864,2
1,0.012658,0.018478,0.000000,0.000000,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.799620,0.811321,0.68205,0.92351,0.922185,0.827737,0.089864,2
2,0.006621,0.007547,0.002285,0.004009,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.804764,0.811321,0.68205,0.92351,0.922185,0.828766,0.089565,1
3,0.001899,0.000930,0.002262,0.002246,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.799620,0.811321,0.68205,0.92351,0.922185,0.827737,0.089864,2


In [20]:
age_input = float(input("Enter Age: "))
salary_input = float(input("Enter Estimated Salary: "))
gender_input = int(input("Gender Male? (1 = Male, 0 = Female): "))




Enter Age: 32
Enter Estimated Salary: 45000
Gender Male? (1 = Male, 0 = Female): 1


In [21]:
# important: apply same scaling
user_data = sc.transform([[age_input, salary_input, gender_input]])


C:\Anaconda3\lib\site-packages\sklearn\base.py:451: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  "X does not have valid feature names, but"


In [22]:
Future_Prediction = grid.predict(user_data)

print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[0]
